In [0]:
import org.apache.spark.sql.functions._
import spark.implicits._
import org.apache.spark.sql.DataFrame
import java.time.{LocalDate, ZoneId}
import java.time.format.DateTimeFormatter
import org.apache.spark.sql.types.{DataType, StructType}

In [0]:
// Obtener parámetros desde widgets
val ruta_adls = dbutils.widgets.get("ruta_adls")
val nombre_archivo = dbutils.widgets.get("nombre_archivo")
val delimitador = dbutils.widgets.get("delimitador")
val encabezado = dbutils.widgets.get("encabezado")
val file_date = dbutils.widgets.get("file_date")
val format_date_filename_in = dbutils.widgets.get("format_date_filename_in")
val campo_particion = dbutils.widgets.get("campo_particion")

In [0]:
// Parsear file_date como LocalDate
val formatterArchivo = DateTimeFormatter.ofPattern(format_date_filename_in)
val fechaActual = LocalDate.parse(file_date, formatterArchivo)

// Formateadores
val formatterRuta = DateTimeFormatter.ofPattern("yyyy/MM/dd")

// Extraer componentes de fecha
val anio = fechaActual.getYear
val mes = f"${fechaActual.getMonthValue}%02d"
val dia = f"${fechaActual.getDayOfMonth}%02d"
val fechaArchivo = file_date

// Ruta original
val rutaOriginal = s"$ruta_adls/landing/$anio/$mes/$dia/$nombre_archivo"

//obtiene schema de archivo
val nombre_json = ruta_adls.split("/").last
val ruta_json = dbutils.fs.head(s"$ruta_adls/$nombre_json.json")
var schema_data = DataType.fromJson(ruta_json).asInstanceOf[StructType]


// Ruta temporal de staging
val rutaTemporalStaging = s"$ruta_adls/tmp_landing/$nombre_archivo"



In [0]:
// Mover archivo original a staging temporal
dbutils.fs.mv(rutaOriginal, rutaTemporalStaging)

// Leer desde la ruta temporal
val df = spark.read
  .option("header", encabezado)
  .option("delimiter", delimitador)
  .schema(schema_data)
  .csv(rutaTemporalStaging)

// Transformar campo de particion a fecha
val dfConFecha = df.withColumn("fecha", to_date(col(campo_particion), format_date_filename_in))
  .withColumn("anio", year(col("fecha")))
  .withColumn("mes", format_string("%02d", month(col("fecha"))))
  .withColumn("dia", format_string("%02d", dayofmonth(col("fecha"))))

// Obtener fechas únicas
val fechasUnicas = dfConFecha.select("anio", "mes", "dia").distinct().collect()

// Procesar cada fecha
fechasUnicas.foreach { fila =>
  val anio = fila.getAs[Int]("anio")
  val mes = fila.getAs[String]("mes")
  val dia = fila.getAs[String]("dia")

  val dfFiltrado = dfConFecha
    .filter(col("anio") === anio && col("mes") === mes && col("dia") === dia)
    .drop("fecha", "anio", "mes", "dia")

  // Ruta temporal por fecha
  val rutaTemporal = s"$ruta_adls/tmp_new_landing/interaccion_chat_temp_${anio}${mes}${dia}"

  // Escribir en temporal
  dfFiltrado.write
    .option("header", encabezado)
    .option("delimiter", delimitador)
    .mode("overwrite")
    .csv(rutaTemporal)

  // Ruta destino final
  val rutaDestino = s"$ruta_adls/landing/$anio/$mes/$dia/"
  dbutils.fs.mkdirs(rutaDestino)

  // Mover y renombrar archivos a .txt
  dbutils.fs.ls(rutaTemporal).foreach { file =>
    if (file.name.endsWith(".csv")) {
      val origen = file.path
      val nuevoNombre = file.name.stripSuffix(".csv") + ".txt"
      val destino = rutaDestino + nuevoNombre
      dbutils.fs.mv(origen, destino)
    }
  }

  // Eliminar carpeta temporal
  dbutils.fs.rm(rutaTemporal, recurse = true)

}

  dbutils.fs.rm(rutaTemporalStaging, recurse = true)
// Ruta base para salida en ADF
val landing_final = s"$ruta_adls/landing/$anio/*"
val result_exit = Map("landing_final" -> landing_final, "status" -> "OK", "timestamp" -> java.time.Instant.now.toString)

val salida_json = result_exit
  .map { case (k, v) => s""""$k":"$v"""" }
  .mkString("{", ",", "}")

dbutils.notebook.exit(salida_json)